In [2]:
import pandas as pd

In [3]:
groundwater = pd.read_csv(
    "../raw/state-ground-water-level-information.csv"
)
dim_state = pd.read_csv(
    "../clean/dim_state.csv"
)
print(groundwater.shape)
print(groundwater.columns.tolist())
print(groundwater.head(10).to_string(index=False))
print("\nData types:")
print(groundwater.dtypes)
print("\nMissing values:")
print(groundwater.isna().sum())

(29, 4)
['State', 'Total Number of Monitored Stations', 'Observed Range of Water Level-min', 'Observed Range of Water Level-max']
            State  Total Number of Monitored Stations  Observed Range of Water Level-min  Observed Range of Water Level-max
   Andhra Pradesh                                 280                             -98.46                             203.54
Arunachal Pradesh                                   4                              -4.19                              -0.52
            Assam                                  68                             -89.66                              30.73
            Bihar                                  22                             -20.98                            1806.51
       Chandigarh                                  19                             -95.15                              73.46
     Chhattisgarh                                 243                            -940.98                              35.03
  

In [4]:
gw_states = set(groundwater["State"].str.strip())
dim_states = set(dim_state["state_name"].str.strip())

print("Missing from groundwater:")
print(sorted(dim_states - gw_states))

print("\nExtra in groundwater:")
print(sorted(gw_states - dim_states))


Missing from groundwater:
['Andaman and Nicobar Islands', 'Dadra and Nagar Haveli and Daman and Diu', 'Jammu and Kashmir', 'Ladakh', 'Lakshadweep', 'Manipur', 'Mizoram', 'Sikkim']

Extra in groundwater:
['Jammu & Kashmir']


In [5]:
from pathlib import Path

PDF_PATH = Path("../raw/Pre-monsoon_WL_1994-2025.pdf")

print("Exists:", PDF_PATH.exists())
print("Path:", PDF_PATH)

Exists: True
Path: ..\raw\Pre-monsoon_WL_1994-2025.pdf


In [6]:
import os

size_gb = os.path.getsize(PDF_PATH) / (1024 ** 3)

print(f"PDF size: {size_gb:.2f} GB")

PDF size: 0.09 GB


In [7]:
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]

    print("Page size:", page.width, "x", page.height)

    text = page.extract_text()
    print(text)

Page size: 841.92004 x 595.32001
Pre-monsoon Depth to Ground Water Level Data (in mbgl) for Unconfined Aquifer
State District Block Village Latitute Longitude Date WL(mbgl)
Andaman And Nicobar Islands North And Middle Andaman Diglipur Durgapur 13.27653 93.02364 30-04-25 0.46
Andaman And Nicobar Islands North And Middle Andaman Diglipur Keralapuram 13.2588 93.01351 30-04-25 1.31
Andaman And Nicobar Islands North And Middle Andaman Diglipur Kishorinagar (Parangara) 13.16506 92.88164 30-04-25 2.21
Andaman And Nicobar Islands North And Middle Andaman Diglipur Laxmipur 13.28734 92.96404 30-04-25 1.46
Andaman And Nicobar Islands North And Middle Andaman Diglipur Milangram 13.31475 92.94309 30-04-25 1.01
Andaman And Nicobar Islands North And Middle Andaman Diglipur Mohanpur 12.95448 92.83567 30-04-25 1.44
Andaman And Nicobar Islands North And Middle Andaman Diglipur Nabagram 13.15474 92.94847 30-04-25 0.55
Andaman And Nicobar Islands North And Middle Andaman Diglipur Rest Camp 12.8328 92.8592

In [8]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]

    tables = page.extract_tables()

    print("Number of tables:", len(tables))

    if tables:
        table = tables[0]

        print("Number of rows:", len(table))
        print("Number of columns:", len(table[0]))

        for row in table[:10]:
            print(row)

Number of tables: 0


In [9]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]

    words = page.extract_words()

    for word in words[:80]:
        print(
            f"{word['text']:<30} "
            f"x0={word['x0']:.1f} "
            f"x1={word['x1']:.1f} "
            f"top={word['top']:.1f}"
        )

Pre-monsoon                    x0=274.8 x1=321.9 top=20.3
Depth                          x0=323.8 x1=345.3 top=20.3
to                             x0=347.3 x1=354.7 top=20.3
Ground                         x0=356.6 x1=383.2 top=20.3
Water                          x0=385.2 x1=407.0 top=20.3
Level                          x0=408.9 x1=427.0 top=20.3
Data                           x0=428.9 x1=445.4 top=20.3
(in                            x0=447.3 x1=456.5 top=20.3
mbgl)                          x0=458.6 x1=478.5 top=20.3
for                            x0=480.5 x1=490.7 top=20.3
Unconfined                     x0=492.5 x1=533.2 top=20.3
Aquifer                        x0=535.1 x1=561.2 top=20.3
State                          x0=20.2 x1=38.3 top=30.7
District                       x0=184.0 x1=209.0 top=30.7
Block                          x0=332.9 x1=351.7 top=30.7
Village                        x0=505.4 x1=528.9 top=30.7
Latitute                       x0=650.3 x1=677.5 top=30.7
Longitude       

In [10]:
import pdfplumber
import pandas as pd
import re


def extract_page(page):
    words = page.extract_words()

    # Group words by their vertical position.
    rows = {}

    for word in words:
        top = round(word["top"], 1)

        if top not in rows:
            rows[top] = []

        rows[top].append(word)

    extracted_rows = []

    for top, words_in_row in rows.items():

        # Sort words from left to right
        words_in_row = sorted(
            words_in_row,
            key=lambda x: x["x0"]
        )

        # Skip header/title rows
        if top <= 35:
            continue

        # Assign words to columns based on x-coordinate
        columns = {
            "State": [],
            "District": [],
            "Block": [],
            "Village": [],
            "Latitude": [],
            "Longitude": [],
            "Date": [],
            "WL_m_bgl": []
        }

        for word in words_in_row:
            x = word["x0"]
            text = word["text"]

            if 15 <= x < 150:
                columns["State"].append(text)

            elif 170 <= x < 315:
                columns["District"].append(text)

            elif 320 <= x < 490:
                columns["Block"].append(text)

            elif 495 <= x < 640:
                columns["Village"].append(text)

            elif 640 <= x < 688:
                columns["Latitude"].append(text)

            elif 688 <= x < 735:
                columns["Longitude"].append(text)

            elif 735 <= x < 790:
                columns["Date"].append(text)

            elif x >= 790:
                columns["WL_m_bgl"].append(text)

        row = {
            column: " ".join(values).strip()
            for column, values in columns.items()
        }

        # Only keep rows that look like actual observations
        if row["Latitude"] and row["Longitude"] and row["Date"]:
            extracted_rows.append(row)

    return pd.DataFrame(extracted_rows)

In [11]:
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]
    df_test = extract_page(page)

print(df_test.shape)
display(df_test.head(10))

(52, 8)


,State,District,Block,Village,Latitude,Longitude,Date,WL_m_bgl
0,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Durgapur,13.27653,93.02364,30-04-25,0.46
1,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Keralapuram,13.2588,93.01351,30-04-25,1.31
2,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Kishorinagar (Parangara),13.16506,92.88164,30-04-25,2.21
3,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Laxmipur,13.28734,92.96404,30-04-25,1.46
4,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Milangram,13.31475,92.94309,30-04-25,1.01
5,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Mohanpur,12.95448,92.83567,30-04-25,1.44
6,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Nabagram,13.15474,92.94847,30-04-25,0.55
7,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Rest Camp,12.8328,92.85923,30-04-25,0.49
8,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Shibpur,13.24783,93.04183,30-04-25,0.14
9,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Sitanagar,13.2261,92.93593,30-04-25,0.32


In [12]:
import pdfplumber
import pandas as pd

test_pages = []

with pdfplumber.open(PDF_PATH) as pdf:

    print("Total pages:", len(pdf.pages))

    for page_number in range(min(10, len(pdf.pages))):

        page = pdf.pages[page_number]

        page_df = extract_page(page)

        page_df["PDF_Page"] = page_number + 1

        test_pages.append(page_df)

df_10_pages = pd.concat(
    test_pages,
    ignore_index=True
)

print("Extracted rows:", len(df_10_pages))
print("Columns:", df_10_pages.columns.tolist())

display(df_10_pages.head())
display(df_10_pages.tail())

Total pages: 10270
Extracted rows: 520
Columns: ['State', 'District', 'Block', 'Village', 'Latitude', 'Longitude', 'Date', 'WL_m_bgl', 'PDF_Page']


,State,District,Block,Village,Latitude,Longitude,Date,WL_m_bgl,PDF_Page
0,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Durgapur,13.27653,93.02364,30-04-25,0.46,1
1,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Keralapuram,13.2588,93.01351,30-04-25,1.31,1
2,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Kishorinagar (Parangara),13.16506,92.88164,30-04-25,2.21,1
3,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Laxmipur,13.28734,92.96404,30-04-25,1.46,1
4,Andaman And Nicobar Islands,North And Middle Andaman,Diglipur,Milangram,13.31475,92.94309,30-04-25,1.01,1


,State,District,Block,Village,Latitude,Longitude,Date,WL_m_bgl,PDF_Page
515,Andhra Pradesh,Palnadu,Machavaram,Pillutla,16.5385,79.9057,25-05-25,2.45,10
516,Andhra Pradesh,Palnadu,Muppalla,Golapadu,16.3067,80.0833,25-05-25,3.07,10
517,Andhra Pradesh,Palnadu,Nadendla,Nadendla,16.1762,80.1843,22-05-25,9.81,10
518,Andhra Pradesh,Palnadu,Narasaraopeta,Jonnalagadda,16.2418,80.0829,22-05-25,2.88,10
519,Andhra Pradesh,Palnadu,Narasaraopeta,Lingamguntla,16.231,80.0318,22-05-25,6.41,10


In [14]:
import pdfplumber
import pandas as pd
from pathlib import Path
import time

OUTPUT_DIR = Path("../raw/groundwater")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "pre_monsoon_1994_2025_raw.parquet"

CHUNK_SIZE = 100
buffer = []

start_time = time.time()

with pdfplumber.open(PDF_PATH) as pdf:

    total_pages = len(pdf.pages)

    print(f"Total pages: {total_pages}")

    for page_number, page in enumerate(pdf.pages, start=1):

        try:
            page_df = extract_page(page)

            if not page_df.empty:
                page_df["PDF_Page"] = page_number
                buffer.append(page_df)

        except Exception as e:
            print(f"ERROR on page {page_number}: {e}")

        # Save every CHUNK_SIZE pages
        if page_number % CHUNK_SIZE == 0:

            if buffer:
                chunk_df = pd.concat(
                    buffer,
                    ignore_index=True
                )

                chunk_file = (
                    OUTPUT_DIR /
                    f"pre_monsoon_pages_{page_number-CHUNK_SIZE+1}_{page_number}.parquet"
                )
                print(f"Saving pages : {page_number-CHUNK_SIZE+1} to {page_number} to {chunk_file}")
                chunk_df.to_parquet(
                    chunk_file,
                    index=False
                )

                buffer = []

            elapsed = time.time() - start_time

            print(
                f"Processed {page_number}/{total_pages} "
                f"pages | {elapsed/60:.1f} min"
            )

    # Save remaining pages
    if buffer:

        chunk_df = pd.concat(
            buffer,
            ignore_index=True
        )

        start_page = (
            total_pages // CHUNK_SIZE
        ) * CHUNK_SIZE + 1

        chunk_file = (
            OUTPUT_DIR /
            f"pre_monsoon_pages_{start_page}_{total_pages}.parquet"
        )

        chunk_df.to_parquet(
            chunk_file,
            index=False
        )

print("Extraction complete.")

Total pages: 10270
Saving pages : 1 to 100 to ..\raw\groundwater\pre_monsoon_pages_1_100.parquet
Processed 100/10270 pages | 0.6 min
Saving pages : 101 to 200 to ..\raw\groundwater\pre_monsoon_pages_101_200.parquet
Processed 200/10270 pages | 1.4 min
Saving pages : 201 to 300 to ..\raw\groundwater\pre_monsoon_pages_201_300.parquet
Processed 300/10270 pages | 2.1 min
Saving pages : 301 to 400 to ..\raw\groundwater\pre_monsoon_pages_301_400.parquet
Processed 400/10270 pages | 3.3 min
Processed 500/10270 pages | 3.9 min
Processed 600/10270 pages | 4.6 min
Processed 700/10270 pages | 5.1 min
Processed 800/10270 pages | 6.0 min
Processed 900/10270 pages | 6.3 min


KeyboardInterrupt: 

In [ ]:
import pdfplumber
import pandas as pd
from pathlib import Path
import time

OUTPUT_DIR = Path("../raw/groundwater")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "pre_monsoon_1994_2025_raw.parquet"

CHUNK_SIZE = 100
buffer = []
PDF_PATH = Path("../raw/August_WL_1994-2025.pdf")
start_time = time.time()

with pdfplumber.open(PDF_PATH) as pdf:

    total_pages = len(pdf.pages)

    print(f"Total pages: {total_pages}")

    for page_number, page in enumerate(pdf.pages, start=1):

        try:
            page_df = extract_page(page)

            if not page_df.empty:
                page_df["PDF_Page"] = page_number
                buffer.append(page_df)

        except Exception as e:
            print(f"ERROR on page {page_number}: {e}")

        # Save every CHUNK_SIZE pages
        if page_number % CHUNK_SIZE == 0:

            if buffer:
                chunk_df = pd.concat(
                    buffer,
                    ignore_index=True
                )

                chunk_file = (
                    OUTPUT_DIR /
                    f"pre_monsoon_pages_{page_number-CHUNK_SIZE+1}_{page_number}.parquet"
                )

                chunk_df.to_parquet(
                    chunk_file,
                    index=False
                )

                buffer = []

            elapsed = time.time() - start_time

            print(
                f"Processed {page_number}/{total_pages} "
                f"pages | {elapsed/60:.1f} min"
            )

    # Save remaining pages
    if buffer:

        chunk_df = pd.concat(
            buffer,
            ignore_index=True
        )

        start_page = (
            total_pages // CHUNK_SIZE
        ) * CHUNK_SIZE + 1

        chunk_file = (
            OUTPUT_DIR /
            f"pre_monsoon_pages_{start_page}_{total_pages}.parquet"
        )

        chunk_df.to_parquet(
            chunk_file,
            index=False
        )

print("Extraction complete.")

In [15]:
import pdfplumber
import pandas as pd

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"

with pdfplumber.open(PDF_PATH) as pdf:

    page = pdf.pages[400]   # PDF page 401

    print("Page size:", page.width, "x", page.height)

    words = page.extract_words()

    print("\nNumber of words:", len(words))

    print("\nFirst 100 extracted words:")
    for w in words[:100]:
        print(
            f"x0={w['x0']:.2f}, "
            f"top={w['top']:.2f}, "
            f"text='{w['text']}'"
        )

Page size: 841.92004 x 595.32001

Number of words: 369

First 100 extracted words:
x0=224.76, top=20.73, text='Pre-monsoon'
x0=291.88, top=20.73, text='Depth'
x0=324.02, top=20.73, text='to'
x0=336.95, top=20.73, text='Ground'
x0=375.90, top=20.73, text='Water'
x0=408.44, top=20.73, text='Level'
x0=435.89, top=20.73, text='Data'
x0=461.04, top=20.73, text='(in'
x0=476.40, top=20.73, text='mbgl)'
x0=506.60, top=20.73, text='for'
x0=523.14, top=20.73, text='Unconfined'
x0=581.58, top=20.73, text='Aquifer'
x0=23.40, top=36.19, text='STATE_UT'
x0=183.50, top=36.19, text='DISTRICT'
x0=323.35, top=36.19, text='BLOCK'
x0=433.17, top=36.19, text='VILLAGE'
x0=572.18, top=36.19, text='LATITUDE'
x0=630.60, top=36.19, text='LONGITUDE'
x0=706.95, top=36.19, text='Date'
x0=762.48, top=36.07, text='WL'
x0=778.90, top=36.07, text='(in'
x0=792.84, top=36.07, text='mbgl)'
x0=23.40, top=50.11, text='Goa'
x0=183.48, top=50.11, text='South'
x0=210.71, top=50.11, text='Goa'
x0=323.40, top=50.11, text='Canac

In [16]:
import pdfplumber
import pandas as pd
import re

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"


# ---------------------------------------------------------
# Header aliases
# ---------------------------------------------------------

HEADER_ALIASES = {
    "State": {
        "STATE",
        "STATE_UT",
        "STATE/UT",
        "STATE_UT_NAME"
    },

    "District": {
        "DISTRICT"
    },

    "Block": {
        "BLOCK"
    },

    "Village": {
        "VILLAGE"
    },

    "Latitude": {
        "LATITUDE",
        "LATITUTE"      # tolerate source typo
    },

    "Longitude": {
        "LONGITUDE",
        "LONGITUTE"
    },

    "Date": {
        "DATE"
    },

    "WL_m_bgl": {
        "WL",
        "WATERLEVEL",
        "WATER_LEVEL"
    }
}


# ---------------------------------------------------------
# Normalize text
# ---------------------------------------------------------

def normalize_header(text):

    text = text.upper().strip()

    # Remove punctuation / brackets
    text = re.sub(
        r"[^A-Z0-9_/]",
        "",
        text
    )

    return text


# ---------------------------------------------------------
# Detect header
# ---------------------------------------------------------

def detect_header(rows):

    for top, row_words in rows.items():

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        detected = {}

        for word in row_words:

            normalized = normalize_header(
                word["text"]
            )

            for column, aliases in HEADER_ALIASES.items():

                if normalized in aliases:

                    detected[column] = word["x0"]

                    break

        # We need at least the important structural
        # columns to consider this a header.
        required = {
            "State",
            "District",
            "Block",
            "Village",
            "Latitude",
            "Longitude"
        }

        if required.issubset(detected):

            return top, detected

    return None, None


# ---------------------------------------------------------
# Extract page
# ---------------------------------------------------------

def extract_page(page):

    words = page.extract_words()

    if not words:
        return pd.DataFrame()

    # ---------------------------------------------
    # Group words by horizontal row
    # ---------------------------------------------

    rows = {}

    for word in words:

        top = round(
            word["top"],
            1
        )

        rows.setdefault(
            top,
            []
        ).append(word)

    # ---------------------------------------------
    # Detect header dynamically
    # ---------------------------------------------

    header_top, header_positions = detect_header(
        rows
    )

    if header_positions is None:

        print("Header not detected")

        return pd.DataFrame()

    # ---------------------------------------------
    # Order columns according to their position
    # ---------------------------------------------

    ordered_columns = sorted(
        header_positions.items(),
        key=lambda x: x[1]
    )

    # ---------------------------------------------
    # Create boundaries from midpoint
    # ---------------------------------------------

    boundaries = {}

    for i, (column, position) in enumerate(
        ordered_columns
    ):

        if i == 0:

            left = 0

        else:

            previous_position = (
                ordered_columns[i - 1][1]
            )

            left = (
                previous_position +
                position
            ) / 2

        if i == len(ordered_columns) - 1:

            right = page.width

        else:

            next_position = (
                ordered_columns[i + 1][1]
            )

            right = (
                position +
                next_position
            ) / 2

        boundaries[column] = (
            left,
            right
        )

    # ---------------------------------------------
    # Extract data rows
    # ---------------------------------------------

    extracted_rows = []

    for top, row_words in rows.items():

        # Ignore title and header
        if top <= header_top:

            continue

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        row = {
            column: []
            for column in header_positions
        }

        for word in row_words:

            x = word["x0"]

            for column, (
                left,
                right
            ) in boundaries.items():

                if left <= x < right:

                    row[column].append(
                        word["text"]
                    )

                    break

        # -----------------------------------------
        # Join words
        # -----------------------------------------

        row = {
            column: " ".join(values).strip()
            for column, values in row.items()
        }

        # -----------------------------------------
        # Validate observation
        # -----------------------------------------

        if (
            row.get("Latitude")
            and row.get("Longitude")
            and row.get("Date")
        ):

            extracted_rows.append(row)

    return pd.DataFrame(
        extracted_rows
    )

In [18]:
import pdfplumber
import pandas as pd
import time
from pathlib import Path
import re


# =========================================================
# CONFIGURATION
# =========================================================

PDF_PATH = Path("../raw/Pre-monsoon_WL_1994-2025.pdf")

OUTPUT_DIR = Path(
    "../raw/groundwater/pre_monsoon_chunks"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

START_PAGE = 1
END_PAGE = 10270
CHUNK_SIZE = 100


# =========================================================
# HEADER ALIASES
# =========================================================

HEADER_ALIASES = {

    "State": {
        "STATE",
        "STATE_UT",
        "STATE/UT",
        "STATE_UT_NAME"
    },

    "District": {
        "DISTRICT"
    },

    "Block": {
        "BLOCK"
    },

    "Village": {
        "VILLAGE"
    },

    "Latitude": {
        "LATITUDE",
        "LATITUTE"
    },

    "Longitude": {
        "LONGITUDE",
        "LONGITUTE"
    },

    "Date": {
        "DATE"
    },

    "WL_m_bgl": {
        "WL",
        "WATERLEVEL",
        "WATER_LEVEL"
    }
}


# =========================================================
# NORMALIZE HEADER TEXT
# =========================================================

def normalize_header(text):

    text = text.upper().strip()

    text = re.sub(
        r"[^A-Z0-9_/]",
        "",
        text
    )

    return text


# =========================================================
# DETECT HEADER
# =========================================================

def detect_header(rows):

    for top, row_words in rows.items():

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        detected = {}

        for word in row_words:

            normalized = normalize_header(
                word["text"]
            )

            for column, aliases in HEADER_ALIASES.items():

                if normalized in aliases:

                    detected[column] = word["x0"]

                    break

        required = {
            "State",
            "District",
            "Block",
            "Village",
            "Latitude",
            "Longitude"
        }

        if required.issubset(detected):

            return top, detected

    return None, None


# =========================================================
# EXTRACT ONE PAGE
# =========================================================

def extract_page(page):

    words = page.extract_words()

    if not words:
        return pd.DataFrame()

    # -----------------------------------------------------
    # Group words by vertical position
    # -----------------------------------------------------

    rows = {}

    for word in words:

        top = round(
            word["top"],
            1
        )

        rows.setdefault(
            top,
            []
        ).append(word)

    # -----------------------------------------------------
    # Detect table header
    # -----------------------------------------------------

    header_top, header_positions = detect_header(
        rows
    )

    if header_positions is None:

        return pd.DataFrame()

    # -----------------------------------------------------
    # Sort columns according to their x-position
    # -----------------------------------------------------

    ordered_columns = sorted(
        header_positions.items(),
        key=lambda x: x[1]
    )

    # -----------------------------------------------------
    # Calculate boundaries dynamically
    # -----------------------------------------------------

    boundaries = {}

    for i, (column, position) in enumerate(
        ordered_columns
    ):

        if i == 0:

            left = 0

        else:

            previous_position = (
                ordered_columns[i - 1][1]
            )

            left = (
                previous_position +
                position
            ) / 2

        if i == len(ordered_columns) - 1:

            right = page.width

        else:

            next_position = (
                ordered_columns[i + 1][1]
            )

            right = (
                position +
                next_position
            ) / 2

        boundaries[column] = (
            left,
            right
        )

    # -----------------------------------------------------
    # Extract rows
    # -----------------------------------------------------

    extracted_rows = []

    for top, row_words in rows.items():

        # Skip title/header
        if top <= header_top:
            continue

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        row = {
            column: []
            for column in header_positions
        }

        for word in row_words:

            x = word["x0"]

            for column, (
                left,
                right
            ) in boundaries.items():

                if left <= x < right:

                    row[column].append(
                        word["text"]
                    )

                    break

        # -------------------------------------------------
        # Join words
        # -------------------------------------------------

        row = {
            column: " ".join(values).strip()
            for column, values in row.items()
        }

        # -------------------------------------------------
        # Validate observation
        # -------------------------------------------------

        if (
            row.get("Latitude")
            and row.get("Longitude")
            and row.get("Date")
            and row.get("WL_m_bgl")
        ):

            extracted_rows.append(row)

    return pd.DataFrame(
        extracted_rows
    )


# =========================================================
# FULL EXTRACTION
# =========================================================

start_time = time.time()

total_rows = 0
successful_pages = 0
failed_pages = []
empty_pages = []


with pdfplumber.open(PDF_PATH) as pdf:

    total_pdf_pages = len(pdf.pages)

    print(
        f"PDF pages available: "
        f"{total_pdf_pages:,}"
    )

    print(
        f"Pages to process: "
        f"{START_PAGE:,} → {END_PAGE:,}"
    )

    print(
        f"Chunk size: {CHUNK_SIZE}"
    )

    print(
        f"Output directory: "
        f"{OUTPUT_DIR}"
    )

    # -----------------------------------------------------
    # Process chunks
    # -----------------------------------------------------

    for chunk_start in range(
        START_PAGE,
        END_PAGE + 1,
        CHUNK_SIZE
    ):

        chunk_end = min(
            chunk_start + CHUNK_SIZE - 1,
            END_PAGE
        )

        print("\n" + "=" * 70)

        print(
            f"Processing pages "
            f"{chunk_start:,} → {chunk_end:,}"
        )

        print("=" * 70)

        chunk_rows = []

        chunk_start_time = time.time()

        # -------------------------------------------------
        # Process individual pages
        # -------------------------------------------------

        for page_number in range(
            chunk_start,
            chunk_end + 1
        ):

            try:

                page = pdf.pages[
                    page_number - 1
                ]

                page_df = extract_page(
                    page
                )

                if not page_df.empty:

                    page_df["PDF_Page"] = (
                        page_number
                    )

                    chunk_rows.append(
                        page_df
                    )

                    successful_pages += 1

                else:

                    empty_pages.append(
                        page_number
                    )

            except Exception as e:

                failed_pages.append(
                    (
                        page_number,
                        str(e)
                    )
                )

        # -------------------------------------------------
        # Save chunk
        # -------------------------------------------------

        if chunk_rows:

            chunk_df = pd.concat(
                chunk_rows,
                ignore_index=True
            )

            output_file = (
                OUTPUT_DIR /
                f"pre_monsoon_pages_"
                f"{chunk_start}_{chunk_end}.parquet"
            )

            chunk_df.to_parquet(
                output_file,
                index=False
            )

            chunk_rows_count = len(
                chunk_df
            )

            total_rows += (
                chunk_rows_count
            )

            chunk_time = (
                time.time()
                - chunk_start_time
            )

            print(
                f"✓ SAVED: "
                f"{output_file.name}"
            )

            print(
                f"  Rows: "
                f"{chunk_rows_count:,}"
            )

            print(
                f"  Time: "
                f"{chunk_time / 60:.2f} min"
            )

        else:

            print(
                f"⚠ No rows extracted "
                f"from {chunk_start}-{chunk_end}"
            )

        # -------------------------------------------------
        # Overall progress
        # -------------------------------------------------

        elapsed = (
            time.time()
            - start_time
        )

        progress = (
            chunk_end
            / END_PAGE
            * 100
        )

        print(
            f"Progress: "
            f"{chunk_end:,}/{END_PAGE:,} "
            f"({progress:.1f}%)"
        )

        print(
            f"Total rows so far: "
            f"{total_rows:,}"
        )

        print(
            f"Elapsed: "
            f"{elapsed / 60:.2f} minutes"
        )


# =========================================================
# FINAL SUMMARY
# =========================================================

total_time = (
    time.time()
    - start_time
)

print("\n")
print("=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)

print(
    f"Pages processed: "
    f"{START_PAGE:,} → {END_PAGE:,}"
)

print(
    f"Successful pages: "
    f"{successful_pages:,}"
)

print(
    f"Empty pages: "
    f"{len(empty_pages):,}"
)

print(
    f"Failed pages: "
    f"{len(failed_pages):,}"
)

print(
    f"Total rows: "
    f"{total_rows:,}"
)

print(
    f"Total time: "
    f"{total_time / 60:.2f} minutes"
)

if failed_pages:

    print("\nFailed pages:")

    for page_number, error in failed_pages[:20]:

        print(
            page_number,
            error
        )

    if len(failed_pages) > 20:

        print(
            f"... and "
            f"{len(failed_pages) - 20} more"
        )

if empty_pages:

    print("\nFirst empty pages:")

    print(
        empty_pages[:30]
    )

PDF pages available: 10,270
Pages to process: 1 → 10,270
Chunk size: 100
Output directory: ..\raw\groundwater\pre_monsoon_chunks

Processing pages 1 → 100
⚠ No rows extracted from 1-100
Progress: 100/10,270 (1.0%)
Total rows so far: 0
Elapsed: 1.44 minutes

Processing pages 101 → 200
⚠ No rows extracted from 101-200
Progress: 200/10,270 (1.9%)
Total rows so far: 0
Elapsed: 2.53 minutes

Processing pages 201 → 300
⚠ No rows extracted from 201-300
Progress: 300/10,270 (2.9%)
Total rows so far: 0
Elapsed: 4.38 minutes

Processing pages 301 → 400
⚠ No rows extracted from 301-400
Progress: 400/10,270 (3.9%)
Total rows so far: 0
Elapsed: 5.36 minutes

Processing pages 401 → 500
⚠ No rows extracted from 401-500
Progress: 500/10,270 (4.9%)
Total rows so far: 0
Elapsed: 6.12 minutes

Processing pages 501 → 600
⚠ No rows extracted from 501-600
Progress: 600/10,270 (5.8%)
Total rows so far: 0
Elapsed: 7.21 minutes

Processing pages 601 → 700
⚠ No rows extracted from 601-700
Progress: 700/10,270 (

KeyboardInterrupt: 

In [19]:
import pdfplumber

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"

with pdfplumber.open(PDF_PATH) as pdf:

    page = pdf.pages[0]

    words = page.extract_words()

    rows = {}

    for word in words:

        top = round(word["top"], 1)

        rows.setdefault(top, []).append(word)

    print("Number of row groups:", len(rows))

    for top, row_words in list(rows.items())[:10]:

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        print("\nTOP:", top)

        print(
            [
                word["text"]
                for word in row_words
            ]
        )

Number of row groups: 54

TOP: 20.3
['Pre-monsoon', 'Depth', 'to', 'Ground', 'Water', 'Level', 'Data', '(in', 'mbgl)', 'for', 'Unconfined', 'Aquifer']

TOP: 30.7
['State', 'District', 'Block', 'Village', 'Latitute', 'Longitude', 'Date', 'WL(mbgl)']

TOP: 41.0
['Andaman', 'And', 'Nicobar', 'Islands', 'North', 'And', 'Middle', 'Andaman', 'Diglipur', 'Durgapur', '13.27653', '93.02364', '30-04-25', '0.46']

TOP: 51.3
['Andaman', 'And', 'Nicobar', 'Islands', 'North', 'And', 'Middle', 'Andaman', 'Diglipur', 'Keralapuram', '13.2588', '93.01351', '30-04-25', '1.31']

TOP: 61.7
['Andaman', 'And', 'Nicobar', 'Islands', 'North', 'And', 'Middle', 'Andaman', 'Diglipur', 'Kishorinagar', '(Parangara)', '13.16506', '92.88164', '30-04-25', '2.21']

TOP: 72.0
['Andaman', 'And', 'Nicobar', 'Islands', 'North', 'And', 'Middle', 'Andaman', 'Diglipur', 'Laxmipur', '13.28734', '92.96404', '30-04-25', '1.46']

TOP: 82.3
['Andaman', 'And', 'Nicobar', 'Islands', 'North', 'And', 'Middle', 'Andaman', 'Diglipur', '

In [20]:
def detect_header(rows):

    header_aliases = {
        "State": {
            "STATE",
            "STATE_UT",
            "STATE/UT"
        },

        "District": {
            "DISTRICT"
        },

        "Block": {
            "BLOCK"
        },

        "Village": {
            "VILLAGE"
        },

        "Latitude": {
            "LATITUDE",
            "LATITUTE"
        },

        "Longitude": {
            "LONGITUDE"
        },

        "Date": {
            "DATE"
        },

        "WL_m_bgl": {
            "WL",
            "WL(MBGL)"
        }
    }

    for top, row_words in rows.items():

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        detected = {}

        for word in row_words:

            text = (
                word["text"]
                .upper()
                .strip()
            )

            # Remove spaces
            text = text.replace(
                " ",
                ""
            )

            for column, aliases in header_aliases.items():

                normalized_aliases = {
                    alias.replace(" ", "")
                    for alias in aliases
                }

                if text in normalized_aliases:

                    detected[column] = (
                        word["x0"]
                    )

                    break

        # Need the main structural columns
        required = {
            "State",
            "District",
            "Block",
            "Village",
            "Latitude",
            "Longitude",
            "Date"
        }

        if required.issubset(
            detected.keys()
        ):

            # WL may be embedded in WL(mbgl)
            # or detected separately.
            if "WL_m_bgl" not in detected:

                # Look for WL(mbgl)
                for word in row_words:

                    text = (
                        word["text"]
                        .upper()
                        .replace(" ", "")
                    )

                    if text.startswith("WL"):

                        detected[
                            "WL_m_bgl"
                        ] = word["x0"]

                        break

            return top, detected

    return None, None

In [22]:
import pdfplumber

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"

with pdfplumber.open(PDF_PATH) as pdf:

    for page_number in [1, 2, 400, 401, 402, 500]:

        page = pdf.pages[
            page_number - 1
        ]

        words = page.extract_words()

        rows = {}

        for word in words:

            top = round(
                word["top"],
                1
            )

            rows.setdefault(
                top,
                []
            ).append(word)

        header_top, header_positions = (
            detect_header(rows)
        )

        print(
            f"\nPage {page_number}"
        )

        print(
            "Header top:",
            header_top
        )

        print(
            "Header positions:",
            header_positions
        )


Page 1
Header top: 30.7
Header positions: {'State': 20.159999250000002, 'District': 183.95999925, 'Block': 332.879997, 'Village': 505.43998725, 'Latitude': 650.2799835000001, 'Longitude': 692.16000375, 'Date': 740.8800202499999, 'WL_m_bgl': 781.199982}

Page 2
Header top: 30.7
Header positions: {'State': 20.159999250000002, 'District': 183.95999925, 'Block': 332.879997, 'Village': 505.43998725, 'Latitude': 650.2799835000001, 'Longitude': 692.16000375, 'Date': 740.8800202499999, 'WL_m_bgl': 781.199982}

Page 400
Header top: 36.2
Header positions: {'State': 23.40000075, 'District': 183.49567159043926, 'Block': 323.35171744044567, 'Village': 433.1683113175974, 'Latitude': 572.1775247461256, 'Longitude': 630.6048373547385, 'Date': 706.9523698061089}

Page 401
Header top: 36.2
Header positions: {'State': 23.40000075, 'District': 183.49567159043926, 'Block': 323.35171744044567, 'Village': 433.1683113175974, 'Latitude': 572.1775247461256, 'Longitude': 630.6048373547385, 'Date': 706.952369806

In [23]:
def extract_page(page):

    words = page.extract_words()

    if not words:
        return pd.DataFrame()

    # -----------------------------------------------------
    # Group words by row
    # -----------------------------------------------------

    rows = {}

    for word in words:

        top = round(
            word["top"],
            1
        )

        rows.setdefault(
            top,
            []
        ).append(word)

    # -----------------------------------------------------
    # Detect header
    # -----------------------------------------------------

    header_top, header_positions = detect_header(
        rows
    )

    if header_positions is None:
        return pd.DataFrame()

    # -----------------------------------------------------
    # IMPORTANT:
    # If WL wasn't detected, create it as the column
    # immediately after Date.
    # -----------------------------------------------------

    if "WL_m_bgl" not in header_positions:

        date_position = header_positions["Date"]

        header_positions["WL_m_bgl"] = (
            date_position + 1
        )

    # -----------------------------------------------------
    # Sort columns by position
    # -----------------------------------------------------

    ordered_columns = sorted(
        header_positions.items(),
        key=lambda x: x[1]
    )

    # -----------------------------------------------------
    # Create dynamic boundaries
    # -----------------------------------------------------

    boundaries = {}

    for i, (column, position) in enumerate(
        ordered_columns
    ):

        if i == 0:

            left = 0

        else:

            previous_position = (
                ordered_columns[i - 1][1]
            )

            left = (
                previous_position +
                position
            ) / 2

        if i == len(ordered_columns) - 1:

            right = page.width

        else:

            next_position = (
                ordered_columns[i + 1][1]
            )

            right = (
                position +
                next_position
            ) / 2

        boundaries[column] = (
            left,
            right
        )

    # -----------------------------------------------------
    # Extract data rows
    # -----------------------------------------------------

    extracted_rows = []

    for top, row_words in rows.items():

        # Ignore title + header
        if top <= header_top:
            continue

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        row = {
            column: []
            for column in header_positions
        }

        # ---------------------------------------------
        # Assign each word to dynamic column
        # ---------------------------------------------

        for word in row_words:

            x = word["x0"]

            for column, (
                left,
                right
            ) in boundaries.items():

                if left <= x < right:

                    row[column].append(
                        word["text"]
                    )

                    break

        # ---------------------------------------------
        # Join words
        # ---------------------------------------------

        row = {
            column: " ".join(values).strip()
            for column, values in row.items()
        }

        # ---------------------------------------------
        # Validate using data patterns
        # ---------------------------------------------

        latitude = row.get(
            "Latitude",
            ""
        )

        longitude = row.get(
            "Longitude",
            ""
        )

        date = row.get(
            "Date",
            ""
        )

        wl = row.get(
            "WL_m_bgl",
            ""
        )

        try:

            float(latitude)
            float(longitude)
            float(wl)

            valid_coordinates = True

        except:

            valid_coordinates = False

        valid_date = bool(
            re.match(
                r"^\d{2}-\d{2}-\d{2,4}$",
                date
            )
        )

        if (
            valid_coordinates
            and valid_date
        ):

            extracted_rows.append(
                row
            )

    return pd.DataFrame(
        extracted_rows
    )

In [24]:
import pdfplumber

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"

with pdfplumber.open(PDF_PATH) as pdf:

    for page_number in [
        1,
        2,
        400,
        401,
        402,
        500
    ]:

        df = extract_page(
            pdf.pages[page_number - 1]
        )

        print("\n" + "=" * 70)

        print(
            f"PAGE {page_number}"
        )

        print("=" * 70)

        print(
            "Shape:",
            df.shape
        )

        print(
            "Columns:",
            df.columns.tolist()
        )

        print(
            df.head(2).to_string(
                index=False
            )
        )


PAGE 1
Shape: (51, 8)
Columns: ['State', 'District', 'Block', 'Village', 'Latitude', 'Longitude', 'Date', 'WL_m_bgl']
                      State                 District    Block     Village Latitude Longitude     Date WL_m_bgl
Andaman And Nicobar Islands North And Middle Andaman Diglipur    Durgapur 13.27653  93.02364 30-04-25     0.46
Andaman And Nicobar Islands North And Middle Andaman Diglipur Keralapuram  13.2588  93.01351 30-04-25     1.31

PAGE 2
Shape: (49, 8)
Columns: ['State', 'District', 'Block', 'Village', 'Latitude', 'Longitude', 'Date', 'WL_m_bgl']
                      State       District      Block    Village Latitude Longitude     Date WL_m_bgl
Andaman And Nicobar Islands South Andamans Ferrargunj    Manjeri 11.53755  92.64795 30-04-25     0.98
Andaman And Nicobar Islands South Andamans Ferrargunj Mannarghat 11.76734  92.70597 30-04-25     1.93

PAGE 400
Shape: (0, 0)
Columns: []
Empty DataFrame
Columns: []
Index: []

PAGE 401
Shape: (0, 0)
Columns: []
Empty DataFra

In [25]:
def extract_page(page):

    words = page.extract_words()

    if not words:
        return pd.DataFrame()

    # =====================================================
    # Group words by row
    # =====================================================

    rows = {}

    for word in words:

        top = round(
            word["top"],
            1
        )

        rows.setdefault(
            top,
            []
        ).append(word)

    # =====================================================
    # Detect header
    # =====================================================

    header_top, header_positions = detect_header(
        rows
    )

    if header_positions is None:
        return pd.DataFrame()

    # =====================================================
    # We only use the actual detected header columns.
    # WL is handled separately from the data row.
    # =====================================================

    base_columns = [
        "State",
        "District",
        "Block",
        "Village",
        "Latitude",
        "Longitude",
        "Date"
    ]

    # Make sure all seven exist
    if not all(
        column in header_positions
        for column in base_columns
    ):
        return pd.DataFrame()

    # =====================================================
    # Sort the 7 known columns
    # =====================================================

    ordered_columns = sorted(
        base_columns,
        key=lambda column:
            header_positions[column]
    )

    # =====================================================
    # Calculate boundaries for the 7 known columns
    # =====================================================

    boundaries = {}

    for i, column in enumerate(
        ordered_columns
    ):

        position = header_positions[column]

        if i == 0:

            left = 0

        else:

            previous_position = (
                header_positions[
                    ordered_columns[i - 1]
                ]
            )

            left = (
                previous_position +
                position
            ) / 2

        if i == len(ordered_columns) - 1:

            # Date extends until the end of
            # the data row. WL will be extracted
            # separately.
            right = page.width

        else:

            next_position = (
                header_positions[
                    ordered_columns[i + 1]
                ]
            )

            right = (
                position +
                next_position
            ) / 2

        boundaries[column] = (
            left,
            right
        )

    # =====================================================
    # Regex patterns
    # =====================================================

    number_pattern = re.compile(
        r"^-?\d+(?:\.\d+)?$"
    )

    date_pattern = re.compile(
        r"^\d{2}-\d{2}-\d{2,4}$"
    )

    # =====================================================
    # Extract rows
    # =====================================================

    extracted_rows = []

    for top, row_words in rows.items():

        if top <= header_top:
            continue

        row_words = sorted(
            row_words,
            key=lambda x: x["x0"]
        )

        # -----------------------------------------------
        # First collect words by the 7 header regions
        # -----------------------------------------------

        row = {
            column: []
            for column in ordered_columns
        }

        for word in row_words:

            x = word["x0"]

            for column in ordered_columns:

                left, right = boundaries[column]

                if left <= x < right:

                    row[column].append(
                        word["text"]
                    )

                    break

        # -----------------------------------------------
        # Join normal columns
        # -----------------------------------------------

        row = {
            column: " ".join(
                values
            ).strip()

            for column, values
            in row.items()
        }

        # -----------------------------------------------
        # The Date region may contain BOTH Date and WL.
        #
        # Example:
        #
        # Date region:
        # 10-05-24 11.4
        #
        # Extract them using patterns.
        # -----------------------------------------------

        date_region = row["Date"]

        tokens = date_region.split()

        date_value = None
        wl_value = None

        for token in tokens:

            if date_pattern.match(token):

                date_value = token

            elif number_pattern.match(token):

                wl_value = token

        # -----------------------------------------------
        # Validate coordinates
        # -----------------------------------------------

        try:

            latitude = float(
                row["Latitude"]
            )

            longitude = float(
                row["Longitude"]
            )

            wl = float(
                wl_value
            )

            valid_numeric = True

        except:

            valid_numeric = False

        # -----------------------------------------------
        # Validate date
        # -----------------------------------------------

        valid_date = (
            date_value is not None
        )

        # -----------------------------------------------
        # Save valid observation
        # -----------------------------------------------

        if (
            valid_numeric
            and valid_date
        ):

            extracted_rows.append({

                "State":
                    row["State"],

                "District":
                    row["District"],

                "Block":
                    row["Block"],

                "Village":
                    row["Village"],

                "Latitude":
                    row["Latitude"],

                "Longitude":
                    row["Longitude"],

                "Date":
                    date_value,

                "WL_m_bgl":
                    wl_value
            })

    return pd.DataFrame(
        extracted_rows
    )

In [26]:
import pdfplumber

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"

with pdfplumber.open(PDF_PATH) as pdf:

    for page_number in [
        1,
        2,
        400,
        401,
        402,
        500
    ]:

        df = extract_page(
            pdf.pages[
                page_number - 1
            ]
        )

        print(
            "\n" + "=" * 70
        )

        print(
            f"PAGE {page_number}"
        )

        print(
            "=" * 70
        )

        print(
            "Shape:",
            df.shape
        )

        if not df.empty:

            print(
                df.head(2).to_string(
                    index=False
                )
            )


PAGE 1
Shape: (51, 8)
                      State                 District    Block     Village Latitude Longitude     Date WL_m_bgl
Andaman And Nicobar Islands North And Middle Andaman Diglipur    Durgapur 13.27653  93.02364 30-04-25     0.46
Andaman And Nicobar Islands North And Middle Andaman Diglipur Keralapuram  13.2588  93.01351 30-04-25     1.31

PAGE 2
Shape: (49, 8)
                      State       District      Block    Village Latitude Longitude     Date WL_m_bgl
Andaman And Nicobar Islands South Andamans Ferrargunj    Manjeri 11.53755  92.64795 30-04-25     0.98
Andaman And Nicobar Islands South Andamans Ferrargunj Mannarghat 11.76734  92.70597 30-04-25     1.93

PAGE 400
Shape: (0, 0)

PAGE 401
Shape: (0, 0)

PAGE 402
Shape: (0, 0)

PAGE 500
Shape: (0, 0)


In [27]:
import pdfplumber
import pandas as pd
import re

PDF_PATH = "../raw/Pre-monsoon_WL_1994-2025.pdf"


DATE_RE = re.compile(
    r"^\d{2}-\d{2}-\d{2,4}$"
)

NUMBER_RE = re.compile(
    r"^-?\d+(?:\.\d+)?$"
)


def is_number(value):
    return bool(NUMBER_RE.match(value))


def is_date(value):
    return bool(DATE_RE.match(value))


def extract_page(page):

    words = page.extract_words()

    if not words:
        return pd.DataFrame()

    # --------------------------------------------------
    # Group words into visual rows
    # --------------------------------------------------

    rows = {}

    for word in words:

        top = round(
            word["top"],
            1
        )

        rows.setdefault(
            top,
            []
        ).append(word)

    # --------------------------------------------------
    # Find the header row
    # --------------------------------------------------

    header_top = None

    for top, row_words in rows.items():

        texts = [
            w["text"].upper()
            for w in row_words
        ]

        joined = " ".join(texts)

        if (
            "STATE" in joined
            and "DISTRICT" in joined
            and "BLOCK" in joined
            and "VILLAGE" in joined
            and "LATITUDE" in joined
        ):
            header_top = top
            break

    if header_top is None:

        return pd.DataFrame()

    extracted = []

    # --------------------------------------------------
    # Process data rows
    # --------------------------------------------------

    for top, row_words in rows.items():

        if top <= header_top:
            continue

        row_words = sorted(
            row_words,
            key=lambda w: w["x0"]
        )

        tokens = [
            w["text"]
            for w in row_words
        ]

        # ----------------------------------------------
        # Find DATE
        # ----------------------------------------------

        date_index = None

        for i, token in enumerate(tokens):

            if is_date(token):

                date_index = i
                break

        if date_index is None:
            continue

        # ----------------------------------------------
        # Need something after DATE for WL
        # ----------------------------------------------

        if date_index + 1 >= len(tokens):

            continue

        wl_token = tokens[
            date_index + 1
        ]

        if not is_number(wl_token):

            continue

        # ----------------------------------------------
        # Latitude and longitude should immediately
        # precede the date.
        # ----------------------------------------------

        if date_index < 2:
            continue

        latitude_token = tokens[
            date_index - 2
        ]

        longitude_token = tokens[
            date_index - 1
        ]

        if not is_number(
            latitude_token
        ):
            continue

        if not is_number(
            longitude_token
        ):
            continue

        # ----------------------------------------------
        # Everything before latitude belongs to:
        #
        # State
        # District
        # Block
        # Village
        #
        # We use the detected column x positions to
        # assign those text tokens.
        # ----------------------------------------------

        latitude_word = row_words[
            date_index - 2
        ]

        longitude_word = row_words[
            date_index - 1
        ]

        date_word = row_words[
            date_index
        ]

        wl_word = row_words[
            date_index + 1
        ]

        # ------------------------------------------------
        # Use actual positions of the data anchors to
        # divide the text before latitude.
        #
        # No hard-coded page coordinates.
        # ------------------------------------------------

        anchor_positions = [
            latitude_word["x0"],
            longitude_word["x0"],
            date_word["x0"],
            wl_word["x0"]
        ]

        # Find the header positions on this page
        # for the first four textual columns.
        #
        # We only use their relative order.
        # ------------------------------------------------

        state_words = []
        district_words = []
        block_words = []
        village_words = []

        # Get header words
        header_words = sorted(
            rows[header_top],
            key=lambda w: w["x0"]
        )

        header_map = {}

        for w in header_words:

            text = (
                w["text"]
                .upper()
                .strip()
            )

            if text in (
                "STATE",
                "STATE_UT"
            ):
                header_map["State"] = w["x0"]

            elif text == "DISTRICT":
                header_map["District"] = w["x0"]

            elif text == "BLOCK":
                header_map["Block"] = w["x0"]

            elif text == "VILLAGE":
                header_map["Village"] = w["x0"]

        required = [
            "State",
            "District",
            "Block",
            "Village"
        ]

        if not all(
            x in header_map
            for x in required
        ):
            continue

        # ------------------------------------------------
        # Calculate boundaries between textual columns
        # using THIS PAGE'S header positions.
        # ------------------------------------------------

        text_columns = [
            (
                "State",
                header_map["State"]
            ),
            (
                "District",
                header_map["District"]
            ),
            (
                "Block",
                header_map["Block"]
            ),
            (
                "Village",
                header_map["Village"]
            )
        ]

        text_columns.sort(
            key=lambda x: x[1]
        )

        boundaries = {}

        for i, (
            column,
            position
        ) in enumerate(text_columns):

            if i == 0:

                left = 0

            else:

                previous = (
                    text_columns[i - 1][1]
                )

                left = (
                    previous + position
                ) / 2

            if i == len(text_columns) - 1:

                right = latitude_word["x0"]

            else:

                next_position = (
                    text_columns[i + 1][1]
                )

                right = (
                    position +
                    next_position
                ) / 2

            boundaries[column] = (
                left,
                right
            )

        # ------------------------------------------------
        # Assign text words
        # ------------------------------------------------

        text_values = {
            "State": [],
            "District": [],
            "Block": [],
            "Village": []
        }

        for word in row_words:

            x = word["x0"]

            # Don't process numeric anchor fields
            if x >= latitude_word["x0"]:
                continue

            for column, (
                left,
                right
            ) in boundaries.items():

                if left <= x < right:

                    text_values[
                        column
                    ].append(
                        word["text"]
                    )

                    break

        # ------------------------------------------------
        # Create final row
        # ------------------------------------------------

        extracted.append({

            "State":
                " ".join(
                    text_values["State"]
                ).strip(),

            "District":
                " ".join(
                    text_values["District"]
                ).strip(),

            "Block":
                " ".join(
                    text_values["Block"]
                ).strip(),

            "Village":
                " ".join(
                    text_values["Village"]
                ).strip(),

            "Latitude":
                latitude_token,

            "Longitude":
                longitude_token,

            "Date":
                tokens[date_index],

            "WL_m_bgl":
                wl_token
        })

    return pd.DataFrame(
        extracted
    )

In [28]:
with pdfplumber.open(PDF_PATH) as pdf:

    for page_number in [
        1,
        2,
        400,
        401,
        402,
        500
    ]:

        df = extract_page(
            pdf.pages[page_number - 1]
        )

        print(
            f"\nPAGE {page_number}"
        )

        print(
            "Shape:",
            df.shape
        )

        if not df.empty:

            print(
                df.head(3).to_string(
                    index=False
                )
            )


PAGE 1
Shape: (0, 0)

PAGE 2
Shape: (0, 0)

PAGE 400
Shape: (0, 0)

PAGE 401
Shape: (0, 0)

PAGE 402
Shape: (0, 0)

PAGE 500
Shape: (0, 0)


In [29]:
import pandas as pd


In [30]:
ground_water_df = pd.read_csv("../raw/groundwater/Pre-monsoon_WL_1994-2025.csv")

In [32]:
ground_water_df.tail(3)

,State,District,Block,Village,Latitude,Longitude,Date,WL_mbgl
394686,West Bengal,Purba Bardhaman,Manteswar,Bamunpara,23.47000,88.19000,1994-04-01,4.91
394687,West Bengal,Purba Bardhaman,Memari-1,Memari,23.18056,88.09222,1994-04-01,3.85
394688,West Bengal,Purba Bardhaman,Memari-1,Pallaroad,23.17389,88.00750,1994-04-01,4.32


In [33]:
import pandas as pd

GROUNDWATER_PATH = "../raw/groundwater/Pre-monsoon_WL_1994-2025.csv"

gw = pd.read_csv(
    GROUNDWATER_PATH
)

print("Shape:", gw.shape)

print("\nColumns:")
print(gw.columns.tolist())

print("\nFirst 5 rows:")
print(gw.head())

print("\nData types:")
print(gw.dtypes)

print("\nMissing values:")
print(gw.isna().sum())

print("\nDuplicate rows:")
print(gw.duplicated().sum())

Shape: (394689, 8)

Columns:
['State', 'District', 'Block', 'Village', 'Latitude', 'Longitude', 'Date', 'WL_mbgl']

First 5 rows:
                         State                  District     Block  \
0  Andaman And Nicobar Islands  North And Middle Andaman  Diglipur   
1  Andaman And Nicobar Islands  North And Middle Andaman  Diglipur   
2  Andaman And Nicobar Islands  North And Middle Andaman  Diglipur   
3  Andaman And Nicobar Islands  North And Middle Andaman  Diglipur   
4  Andaman And Nicobar Islands  North And Middle Andaman  Diglipur   

                    Village  Latitude  Longitude        Date  WL_mbgl  
0                  Durgapur  13.27653   93.02364  2025-04-30     0.46  
1               Keralapuram  13.25880   93.01351  2025-04-30     1.31  
2  Kishorinagar (Parangara)  13.16506   92.88164  2025-04-30     2.21  
3                  Laxmipur  13.28734   92.96404  2025-04-30     1.46  
4                 Milangram  13.31475   92.94309  2025-04-30     1.01  

Data types:
Stat

In [34]:
# =========================================================
# STEP 2 — DUPLICATE + MISSING VALUE INVESTIGATION
# =========================================================

# 1. Show the rows with missing groundwater level
print("Missing WL rows:")
print(
    gw[gw["WL_mbgl"].isna()].to_string(index=False)
)


# 2. Count exact duplicate rows
duplicate_rows = gw[
    gw.duplicated(
        keep=False
    )
].sort_values(
    [
        "State",
        "District",
        "Block",
        "Village",
        "Date"
    ]
)

print("\nTotal rows involved in exact duplicates:")
print(len(duplicate_rows))


# 3. Show some duplicate examples
print("\nFirst 20 duplicate rows:")
print(
    duplicate_rows.head(20).to_string(
        index=False
    )
)


# 4. Check whether duplicate rows are truly identical
print("\nExact duplicate count:")
print(
    gw.duplicated().sum()
)


# 5. Check whether duplicates occur within the
# same location/date but have different WL values

location_date_cols = [
    "State",
    "District",
    "Block",
    "Village",
    "Latitude",
    "Longitude",
    "Date"
]

location_date_duplicates = (
    gw[
        gw.duplicated(
            subset=location_date_cols,
            keep=False
        )
    ]
    .sort_values(location_date_cols)
)

print(
    "\nRows sharing the same "
    "location + date:"
)

print(
    len(location_date_duplicates)
)

print(
    location_date_duplicates.head(30).to_string(
        index=False
    )
)

Missing WL rows:
      State District    Block Village  Latitude  Longitude       Date  WL_mbgl
Uttarakhand Dehradun Sahaspur  Jhajra  30.43861   77.74778 2019-05-01      NaN

Total rows involved in exact duplicates:
498

First 20 duplicate rows:
         State              District     Block         Village  Latitude  Longitude       Date  WL_mbgl
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2011-05-01     8.74
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2011-05-01     8.74
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2012-05-01     9.60
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2012-05-01     9.60
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2016-05-01     9.09
Andhra Pradesh Alluri Sitharama Raju Nellipaka    Lakshmipuram  17.71967   81.04836 2016-05-01     9.09
Andhra Pradesh Alluri Sit

In [35]:
# =========================================================
# STEP 3 — BASIC CLEANING
# =========================================================

gw_clean = gw.copy()

print("Original rows:", len(gw_clean))


# ---------------------------------------------------------
# 1. Parse date
# ---------------------------------------------------------

gw_clean["Date"] = pd.to_datetime(
    gw_clean["Date"],
    errors="coerce"
)


# ---------------------------------------------------------
# 2. Ensure numeric columns are numeric
# ---------------------------------------------------------

gw_clean["Latitude"] = pd.to_numeric(
    gw_clean["Latitude"],
    errors="coerce"
)

gw_clean["Longitude"] = pd.to_numeric(
    gw_clean["Longitude"],
    errors="coerce"
)

gw_clean["WL_mbgl"] = pd.to_numeric(
    gw_clean["WL_mbgl"],
    errors="coerce"
)


# ---------------------------------------------------------
# 3. Check parsing failures
# ---------------------------------------------------------

print("\nDate parsing failures:")
print(
    gw_clean["Date"].isna().sum()
)

print("\nLatitude parsing failures:")
print(
    gw_clean["Latitude"].isna().sum()
)

print("\nLongitude parsing failures:")
print(
    gw_clean["Longitude"].isna().sum()
)

print("\nWL missing:")
print(
    gw_clean["WL_mbgl"].isna().sum()
)


# ---------------------------------------------------------
# 4. Remove ONLY exact duplicates
# ---------------------------------------------------------

before = len(gw_clean)

gw_clean = gw_clean.drop_duplicates(
    keep="first"
).reset_index(
    drop=True
)

after = len(gw_clean)

print("\nExact duplicates removed:")
print(before - after)

print("Rows after deduplication:")
print(after)


# ---------------------------------------------------------
# 5. Create temporal features
# ---------------------------------------------------------

gw_clean["Year"] = (
    gw_clean["Date"].dt.year
)

gw_clean["Month"] = (
    gw_clean["Date"].dt.month
)


# ---------------------------------------------------------
# 6. Season
# ---------------------------------------------------------

gw_clean["Season"] = "Pre-Monsoon"

print("\nFinal shape:")
print(
    gw_clean.shape
)

print("\nColumns:")
print(
    gw_clean.columns.tolist()
)

print("\nDate range:")
print(
    gw_clean["Date"].min(),
    "→",
    gw_clean["Date"].max()
)

Original rows: 394689

Date parsing failures:
0

Latitude parsing failures:
0

Longitude parsing failures:
0

WL missing:
1

Exact duplicates removed:
249
Rows after deduplication:
394440

Final shape:
(394440, 11)

Columns:
['State', 'District', 'Block', 'Village', 'Latitude', 'Longitude', 'Date', 'WL_mbgl', 'Year', 'Month', 'Season']

Date range:
1994-04-01 00:00:00 → 2025-05-30 00:00:00


In [36]:
# =========================================================
# CHECK STATE / DISTRICT NAME CONSISTENCY
# =========================================================

print("\nNumber of states:")
print(
    gw_clean["State"].nunique()
)

print("\nStates:")
print(
    sorted(
        gw_clean["State"].dropna().unique()
    )
)


print("\nNumber of districts:")
print(
    gw_clean["District"].nunique()
)


print("\nDistricts by state:")
district_counts = (
    gw_clean
    .groupby("State")["District"]
    .nunique()
    .sort_values(
        ascending=False
    )
)

print(
    district_counts
)


Number of states:
35

States:
['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli & Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'The Dadra And Nagar Haveli And Daman And Diu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

Number of districts:
715

Districts by state:
State
Uttar Pradesh                                   75
Madhya Pradesh                                  56
Rajasthan                                       50
Tamil Nadu                                      38
Bihar                                           38
Maharashtra                                     36
Chhattisgarh                                    33
Gu

In [37]:
# =========================================================
# STEP 4 — FIND POSSIBLE STATE NAME VARIANTS
# =========================================================

import re


def normalize_for_comparison(text):

    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    # Replace &, punctuation with spaces
    text = text.replace("&", " and ")

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Collapse whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


state_check = (
    gw_clean[
        ["State"]
    ]
    .drop_duplicates()
    .copy()
)

state_check["Comparison_Name"] = (
    state_check["State"]
    .apply(normalize_for_comparison)
)

print(
    state_check
    .sort_values("Comparison_Name")
    .to_string(index=False)
)

                                       State                              Comparison_Name
                 Andaman And Nicobar Islands                  andaman and nicobar islands
                              Andhra Pradesh                               andhra pradesh
                           Arunachal Pradesh                            arunachal pradesh
                                       Assam                                        assam
                                       Bihar                                        bihar
                                  Chandigarh                                   chandigarh
                                Chhattisgarh                                 chhattisgarh
      Dadra And Nagar Haveli & Daman And Diu     dadra and nagar haveli and daman and diu
                                       Delhi                                        delhi
                                         Goa                                          goa
          

In [38]:
# =========================================================
# FIND POSSIBLE DISTRICT NAME VARIANTS
# =========================================================

district_check = (
    gw_clean[
        ["State", "District"]
    ]
    .drop_duplicates()
    .copy()
)

district_check["Comparison_State"] = (
    district_check["State"]
    .apply(normalize_for_comparison)
)

district_check["Comparison_District"] = (
    district_check["District"]
    .apply(normalize_for_comparison)
)


# Find cases where the same normalized
# state + district has multiple source names.

possible_variants = (
    district_check
    .groupby(
        [
            "Comparison_State",
            "Comparison_District"
        ]
    )
    .filter(
        lambda x: len(x) > 1
    )
    .sort_values(
        [
            "Comparison_State",
            "Comparison_District"
        ]
    )
)


print(
    "Possible district naming variants:"
)

print(
    possible_variants.to_string(
        index=False
    )
)

Possible district naming variants:
            State  District  Comparison_State Comparison_District
Jammu And Kashmir Bandipora jammu and kashmir           bandipora
Jammu and Kashmir Bandipora jammu and kashmir           bandipora
Jammu And Kashmir Baramulla jammu and kashmir           baramulla
Jammu and Kashmir Baramulla jammu and kashmir           baramulla
Jammu And Kashmir Ganderbal jammu and kashmir           ganderbal
Jammu and Kashmir Ganderbal jammu and kashmir           ganderbal
Jammu And Kashmir     Jammu jammu and kashmir               jammu
Jammu and Kashmir     Jammu jammu and kashmir               jammu
Jammu And Kashmir    Kathua jammu and kashmir              kathua
Jammu and Kashmir    Kathua jammu and kashmir              kathua
Jammu And Kashmir   Kupwara jammu and kashmir             kupwara
Jammu and Kashmir   Kupwara jammu and kashmir             kupwara
Jammu And Kashmir    Poonch jammu and kashmir              poonch
Jammu and Kashmir    Poonch jammu and kas

In [39]:
# =========================================================
# STEP 5 — STANDARDIZE STATE / DISTRICT TEXT
# =========================================================

def standardize_text(text):

    if pd.isna(text):
        return text

    text = str(text).strip()

    # Normalize ampersand
    text = text.replace("&", "and")

    # Remove repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Consistent capitalization
    text = text.title()

    return text


gw_clean["State_Standard"] = (
    gw_clean["State"]
    .apply(standardize_text)
)

gw_clean["District_Standard"] = (
    gw_clean["District"]
    .apply(standardize_text)
)


print("Standardized states:")
print(
    sorted(
        gw_clean["State_Standard"]
        .dropna()
        .unique()
    )
)

print(
    "\nNumber of standardized states:",
    gw_clean["State_Standard"].nunique()
)

print(
    "\nNumber of standardized districts:",
    gw_clean["District_Standard"].nunique()
)

Standardized states:
['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli And Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'The Dadra And Nagar Haveli And Daman And Diu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

Number of standardized states: 34

Number of standardized districts: 714


In [40]:
state_summary = (
    gw_clean
    .groupby(
        "State_Standard"
    )
    .agg(
        Observations=(
            "WL_mbgl",
            "size"
        ),
        Districts=(
            "District_Standard",
            "nunique"
        ),
        Min_Year=(
            "Year",
            "min"
        ),
        Max_Year=(
            "Year",
            "max"
        )
    )
    .sort_values(
        "Observations",
        ascending=False
    )
)

print(
    state_summary.to_string()
)

                                              Observations  Districts  Min_Year  Max_Year
State_Standard                                                                           
Maharashtra                                          40660         36      1994      2025
Madhya Pradesh                                       34301         55      1994      2025
Odisha                                               32772         30      1994      2025
Rajasthan                                            30135         50      1994      2025
Uttar Pradesh                                        29568         75      1994      2025
Karnataka                                            28602         31      1994      2025
Kerala                                               26135         14      1994      2025
Gujarat                                              23740         33      1994      2025
Andhra Pradesh                                       19612         26      1994      2025
Chhattisga

In [41]:
# =========================================================
# STEP 6 — FIX CONFIRMED STATE NAME VARIANT
# =========================================================

gw_clean["State_Standard"] = (
    gw_clean["State_Standard"]
    .replace({
        "The Dadra And Nagar Haveli And Daman And Diu":
            "Dadra And Nagar Haveli And Daman And Diu"
    })
)

print(
    "Standardized states:",
    gw_clean["State_Standard"].nunique()
)

print("\nDadra/Daman observations:")
print(
    gw_clean[
        gw_clean["State_Standard"]
        == "Dadra And Nagar Haveli And Daman And Diu"
    ]
    .groupby("Year")
    .size()
)

Standardized states: 33

Dadra/Daman observations:
Year
1994     4
1995     4
1996     3
1997     4
1998     3
1999     3
2000     3
2001     1
2002     4
2003     3
2004     4
2005     5
2006    10
2007     4
2008    10
2009     8
2010     9
2011    11
2012    10
2013    11
2014    22
2015    22
2016    28
2017    30
2018    27
2019    29
2020    16
2021     6
2022    32
2023    26
2024    41
2025    37
dtype: int64


In [42]:
# =========================================================
# STEP 7 — MONITORING LOCATION ANALYSIS
# =========================================================

location_cols = [
    "State_Standard",
    "District_Standard",
    "Latitude",
    "Longitude"
]

# Unique physical-ish monitoring locations
station_summary = (
    gw_clean
    .groupby(location_cols)
    .agg(
        Observations=("WL_mbgl", "size"),
        Years=("Year", "nunique"),
        Min_Year=("Year", "min"),
        Max_Year=("Year", "max"),
        Villages=("Village", "nunique"),
        Blocks=("Block", "nunique"),
        Dates=("Date", "nunique")
    )
    .reset_index()
)

print("Potential monitoring locations:")
print(len(station_summary))

print("\nFirst 20:")
print(
    station_summary.head(20).to_string(
        index=False
    )
)

Potential monitoring locations:
41978

First 20:
             State_Standard        District_Standard  Latitude  Longitude  Observations  Years  Min_Year  Max_Year  Villages  Blocks  Dates
Andaman And Nicobar Islands North And Middle Andaman 12.168260  92.766920             1      1      2025      2025         1       1      1
Andaman And Nicobar Islands North And Middle Andaman 12.168280  92.767000            19     19      2004      2024         1       1     19
Andaman And Nicobar Islands North And Middle Andaman 12.186820  92.791870             1      1      2025      2025         1       1      1
Andaman And Nicobar Islands North And Middle Andaman 12.186870  92.791930            10     10      2013      2024         1       1     10
Andaman And Nicobar Islands North And Middle Andaman 12.328020  92.790110            19     19      2004      2024         1       1     19
Andaman And Nicobar Islands North And Middle Andaman 12.328100  92.790020             1      1      2025      2

In [43]:
print("\nLocations with observations in multiple years:")

print(
    (
        station_summary["Years"] > 1
    ).sum()
)

print(
    "\nLocations with 10+ years of observations:"
)

print(
    (
        station_summary["Years"] >= 10
    ).sum()
)

print(
    "\nLocations with 20+ years of observations:"
)

print(
    (
        station_summary["Years"] >= 20
    ).sum()
)


Locations with observations in multiple years:
28401

Locations with 10+ years of observations:
15969

Locations with 20+ years of observations:
7525


In [44]:
multi_village_locations = (
    station_summary[
        station_summary["Villages"] > 1
    ]
)

multi_block_locations = (
    station_summary[
        station_summary["Blocks"] > 1
    ]
)

print(
    "\nCoordinates associated with multiple villages:",
    len(multi_village_locations)
)

print(
    "Coordinates associated with multiple blocks:",
    len(multi_block_locations)
)

print("\nExamples:")
print(
    multi_village_locations
    .head(20)
    .to_string(index=False)
)


Coordinates associated with multiple villages: 368
Coordinates associated with multiple blocks: 255

Examples:
             State_Standard           District_Standard  Latitude  Longitude  Observations  Years  Min_Year  Max_Year  Villages  Blocks  Dates
Andaman And Nicobar Islands              South Andamans 11.537550  92.647950             2      1      2025      2025         2       1      1
             Andhra Pradesh                   Annamayya 14.097778  78.554167             2      1      2001      2001         2       1      2
             Andhra Pradesh                   Annamayya 14.097780  78.554170            14      7      2002      2008         2       1      9
             Andhra Pradesh                     Bapatla 15.993060  80.026390             5      4      2000      2004         2       1      4
             Andhra Pradesh Dr. B.R. Ambedkar Konaseema 16.515000  81.934720            25     11      1997      2008         3       1     19
             Andhra Pradesh Dr

In [45]:
# =========================================================
# STEP 8 — STATION-YEAR OBSERVATION ANALYSIS
# =========================================================

station_cols = [
    "State_Standard",
    "District_Standard",
    "Latitude",
    "Longitude"
]

# ---------------------------------------------------------
# 1. Create station-year summary
# ---------------------------------------------------------

station_year = (
    gw_clean
    .groupby(station_cols + ["Year"])
    .agg(
        Observations=("WL_mbgl", "size"),
        Valid_WL=("WL_mbgl", "count"),
        Mean_WL=("WL_mbgl", "mean"),
        Min_WL=("WL_mbgl", "min"),
        Max_WL=("WL_mbgl", "max"),
        Unique_WL=("WL_mbgl", "nunique")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 2. Overall station-year statistics
# ---------------------------------------------------------

print("=" * 70)
print("STATION-YEAR ANALYSIS")
print("=" * 70)

print(
    f"\nTotal station-year records: "
    f"{len(station_year):,}"
)

print(
    "\nObservations per station-year:"
)

print(
    station_year["Observations"].describe()
)


# ---------------------------------------------------------
# 3. Frequency of observations per station-year
# ---------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "OBSERVATION FREQUENCY"
)

print("=" * 70)

print(
    station_year["Observations"]
    .value_counts()
    .sort_index()
    .head(20)
)


# ---------------------------------------------------------
# 4. Station-years with multiple observations
# ---------------------------------------------------------

multi_obs = station_year[
    station_year["Observations"] > 1
]

print(
    "\n" + "=" * 70
)

print(
    "MULTIPLE OBSERVATIONS PER STATION-YEAR"
)

print("=" * 70)

print(
    f"Station-years with >1 observation: "
    f"{len(multi_obs):,}"
)

print(
    f"Percentage of station-years: "
    f"{len(multi_obs) / len(station_year) * 100:.2f}%"
)


# ---------------------------------------------------------
# 5. Multiple observations with different WL values
# ---------------------------------------------------------

different_measurements = station_year[
    (
        station_year["Observations"] > 1
    )
    &
    (
        station_year["Unique_WL"] > 1
    )
]

print(
    "\n" + "=" * 70
)

print(
    "MULTIPLE DIFFERENT WL VALUES"
)

print("=" * 70)

print(
    f"Station-years with multiple "
    f"different WL values: "
    f"{len(different_measurements):,}"
)

print(
    f"Percentage: "
    f"{len(different_measurements) / len(station_year) * 100:.2f}%"
)


# ---------------------------------------------------------
# 6. Show examples
# ---------------------------------------------------------

if len(different_measurements) > 0:

    print(
        "\nExamples of station-years "
        "with different WL measurements:"
    )

    print(
        different_measurements
        .sort_values(
            "Observations",
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )


# ---------------------------------------------------------
# 7. Distribution of number of distinct WL values
# ---------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "DISTINCT WL VALUES PER STATION-YEAR"
)

print("=" * 70)

print(
    station_year["Unique_WL"]
    .value_counts()
    .sort_index()
    .head(20)
)

STATION-YEAR ANALYSIS

Total station-year records: 390,667

Observations per station-year:
count    390667.000000
mean          1.009658
std           0.104554
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Observations, dtype: float64

OBSERVATION FREQUENCY
Observations
1    387124
2      3350
3       156
4        37
Name: count, dtype: int64

MULTIPLE OBSERVATIONS PER STATION-YEAR
Station-years with >1 observation: 3,543
Percentage of station-years: 0.91%

MULTIPLE DIFFERENT WL VALUES
Station-years with multiple different WL values: 3,444
Percentage: 0.88%

Examples of station-years with different WL measurements:
State_Standard District_Standard  Latitude  Longitude  Year  Observations  Valid_WL  Mean_WL  Min_WL  Max_WL  Unique_WL
        Punjab           Patiala  30.18472   76.51333  2021             4         4  38.7250   37.40   39.30          4
   Maharashtra         Dharashiv  18.18750   75.99028  1994   

In [46]:
# =========================================================
# STEP 9 — BUILD STATION × YEAR DATASET
# =========================================================

station_cols = [
    "State_Standard",
    "District_Standard",
    "Latitude",
    "Longitude",
    "Year"
]

station_year_clean = (
    gw_clean
    .groupby(
        station_cols,
        as_index=False
    )
    .agg(
        GW_Mean=("WL_mbgl", "mean"),
        GW_Min=("WL_mbgl", "min"),
        GW_Max=("WL_mbgl", "max"),
        GW_Std=("WL_mbgl", "std"),
        Observation_Count=("WL_mbgl", "count")
    )
)

# ---------------------------------------------------------
# Fill std for stations with only one observation
# ---------------------------------------------------------

station_year_clean["GW_Std"] = (
    station_year_clean["GW_Std"]
    .fillna(0)
)

print("=" * 70)
print("STATION × YEAR DATASET")
print("=" * 70)

print(
    "\nShape:",
    station_year_clean.shape
)

print(
    "\nColumns:"
)

print(
    station_year_clean.columns.tolist()
)

print(
    "\nFirst 10 rows:"
)

print(
    station_year_clean
    .head(10)
    .to_string(index=False)
)

print(
    "\nObservation count distribution:"
)

print(
    station_year_clean[
        "Observation_Count"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nMissing values:"
)

print(
    station_year_clean.isna().sum()
)

STATION × YEAR DATASET

Shape: (390667, 10)

Columns:
['State_Standard', 'District_Standard', 'Latitude', 'Longitude', 'Year', 'GW_Mean', 'GW_Min', 'GW_Max', 'GW_Std', 'Observation_Count']

First 10 rows:
             State_Standard        District_Standard  Latitude  Longitude  Year  GW_Mean  GW_Min  GW_Max  GW_Std  Observation_Count
Andaman And Nicobar Islands North And Middle Andaman  12.16826   92.76692  2025     1.02    1.02    1.02     0.0                  1
Andaman And Nicobar Islands North And Middle Andaman  12.16828   92.76700  2004     5.30    5.30    5.30     0.0                  1
Andaman And Nicobar Islands North And Middle Andaman  12.16828   92.76700  2005     1.54    1.54    1.54     0.0                  1
Andaman And Nicobar Islands North And Middle Andaman  12.16828   92.76700  2006     0.60    0.60    0.60     0.0                  1
Andaman And Nicobar Islands North And Middle Andaman  12.16828   92.76700  2007     3.77    3.77    3.77     0.0                  1
And

In [47]:
# =========================================================
# STEP 10 — DISTRICT × YEAR GROUNDWATER AGGREGATION
# =========================================================

district_year_gw = (
    station_year_clean
    .groupby(
        [
            "State_Standard",
            "District_Standard",
            "Year"
        ],
        as_index=False
    )
    .agg(
        GW_Mean=("GW_Mean", "mean"),
        GW_Median=("GW_Mean", "median"),
        GW_Min=("GW_Mean", "min"),
        GW_Max=("GW_Mean", "max"),
        GW_Std=("GW_Mean", "std"),
        GW_Q25=("GW_Mean", lambda x: x.quantile(0.25)),
        GW_Q75=("GW_Mean", lambda x: x.quantile(0.75)),
        Station_Count=("Latitude", "count"),
        Total_Observations=("Observation_Count", "sum")
    )
)

# ---------------------------------------------------------
# Single-station districts have no spatial std
# ---------------------------------------------------------

district_year_gw["GW_Std"] = (
    district_year_gw["GW_Std"]
    .fillna(0)
)

print("=" * 70)
print("DISTRICT × YEAR GROUNDWATER DATASET")
print("=" * 70)

print(
    "\nShape:",
    district_year_gw.shape
)

print(
    "\nColumns:"
)

print(
    district_year_gw.columns.tolist()
)

print(
    "\nFirst 15 rows:"
)

print(
    district_year_gw
    .head(15)
    .to_string(index=False)
)

print(
    "\nMissing values:"
)

print(
    district_year_gw.isna().sum()
)

print(
    "\nYear range:"
)

print(
    district_year_gw["Year"].min(),
    "to",
    district_year_gw["Year"].max()
)

print(
    "\nStation count statistics:"
)

print(
    district_year_gw["Station_Count"].describe()
)

print(
    "\nDistrict-year records by year:"
)

print(
    district_year_gw
    .groupby("Year")
    .size()
    .tail(15)
)

DISTRICT × YEAR GROUNDWATER DATASET

Shape: (20559, 12)

Columns:
['State_Standard', 'District_Standard', 'Year', 'GW_Mean', 'GW_Median', 'GW_Min', 'GW_Max', 'GW_Std', 'GW_Q25', 'GW_Q75', 'Station_Count', 'Total_Observations']

First 15 rows:
             State_Standard        District_Standard  Year  GW_Mean  GW_Median  GW_Min  GW_Max   GW_Std  GW_Q25  GW_Q75  Station_Count  Total_Observations
Andaman And Nicobar Islands North And Middle Andaman  2000 1.840000      1.430    0.22    4.06 1.423528  0.8600  2.7250              7                   7
Andaman And Nicobar Islands North And Middle Andaman  2001 2.678571      2.780    1.01    4.26 1.138807  1.9400  3.4100              7                   7
Andaman And Nicobar Islands North And Middle Andaman  2003 1.980000      1.700    0.80    3.60 0.970636  1.4700  2.4100              7                   7
Andaman And Nicobar Islands North And Middle Andaman  2004 3.367037      3.220    1.37    8.96 1.435971  2.4650  3.7550             27   

In [48]:
# =========================================================
# STEP 11 — GROUNDWATER TEMPORAL FEATURES
# =========================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Helper: calculate linear trend
# ---------------------------------------------------------

def calculate_trend(group):
    """
    Linear trend of groundwater level against year.
    Positive slope  -> groundwater depth increasing
    Negative slope  -> groundwater depth decreasing
    """

    group = group.dropna(
        subset=["Year", "GW_Mean"]
    )

    if len(group) < 2:
        return np.nan

    x = group["Year"].to_numpy()
    y = group["GW_Mean"].to_numpy()

    return np.polyfit(x, y, 1)[0]


# ---------------------------------------------------------
# Feature extraction helper
# ---------------------------------------------------------

def period_features(df, start_year, end_year, prefix):

    period = df[
        df["Year"].between(
            start_year,
            end_year
        )
    ].copy()

    grouped = (
        period
        .groupby(
            [
                "State_Standard",
                "District_Standard"
            ]
        )
    )

    features = grouped.agg(
        **{
            f"{prefix}_Mean": (
                "GW_Mean",
                "mean"
            ),
            f"{prefix}_Median": (
                "GW_Median",
                "mean"
            ),
            f"{prefix}_Std": (
                "GW_Mean",
                "std"
            ),
            f"{prefix}_Min": (
                "GW_Mean",
                "min"
            ),
            f"{prefix}_Max": (
                "GW_Mean",
                "max"
            ),
            f"{prefix}_Years_Available": (
                "Year",
                "nunique"
            ),
            f"{prefix}_Avg_Stations": (
                "Station_Count",
                "mean"
            )
        }
    ).reset_index()

    # Linear trend
    trend = (
        grouped
        .apply(
            calculate_trend,
            include_groups=False
        )
        .reset_index(
            name=f"{prefix}_Trend"
        )
    )

    features = features.merge(
        trend,
        on=[
            "State_Standard",
            "District_Standard"
        ],
        how="left"
    )

    return features


# =========================================================
# CREATE PERIOD FEATURES
# =========================================================

gw_1994_2025 = period_features(
    district_year_gw,
    1994,
    2025,
    "GW_1994_2025"
)

gw_2015_2025 = period_features(
    district_year_gw,
    2015,
    2025,
    "GW_2015_2025"
)

gw_2021_2025 = period_features(
    district_year_gw,
    2021,
    2025,
    "GW_2021_2025"
)


# =========================================================
# LATEST 2025 FEATURES
# =========================================================

gw_2025 = (
    district_year_gw[
        district_year_gw["Year"] == 2025
    ][
        [
            "State_Standard",
            "District_Standard",
            "GW_Mean",
            "GW_Median",
            "GW_Min",
            "GW_Max",
            "GW_Std",
            "Station_Count",
            "Total_Observations"
        ]
    ]
    .rename(
        columns={
            "GW_Mean": "GW_Latest_2025",
            "GW_Median": "GW_Latest_Median_2025",
            "GW_Min": "GW_Latest_Min_2025",
            "GW_Max": "GW_Latest_Max_2025",
            "GW_Std": "GW_Latest_Std_2025",
            "Station_Count": "GW_Stations_2025",
            "Total_Observations": "GW_Observations_2025"
        }
    )
)


# =========================================================
# COMBINE FEATURES
# =========================================================

gw_features = gw_1994_2025.merge(
    gw_2015_2025,
    on=[
        "State_Standard",
        "District_Standard"
    ],
    how="outer"
)

gw_features = gw_features.merge(
    gw_2021_2025,
    on=[
        "State_Standard",
        "District_Standard"
    ],
    how="outer"
)

gw_features = gw_features.merge(
    gw_2025,
    on=[
        "State_Standard",
        "District_Standard"
    ],
    how="outer"
)


# =========================================================
# EARLY VS RECENT CHANGE
# =========================================================

# Earlier 5-year period
gw_2015_2019 = (
    district_year_gw[
        district_year_gw["Year"].between(
            2015,
            2019
        )
    ]
    .groupby(
        [
            "State_Standard",
            "District_Standard"
        ]
    )["GW_Mean"]
    .mean()
    .reset_index(
        name="GW_Mean_2015_2019"
    )
)

# Recent 5-year period
gw_2021_2025_mean = (
    district_year_gw[
        district_year_gw["Year"].between(
            2021,
            2025
        )
    ]
    .groupby(
        [
            "State_Standard",
            "District_Standard"
        ]
    )["GW_Mean"]
    .mean()
    .reset_index(
        name="GW_Mean_2021_2025"
    )
)

gw_features = gw_features.merge(
    gw_2015_2019,
    on=[
        "State_Standard",
        "District_Standard"
    ],
    how="left"
)

gw_features = gw_features.merge(
    gw_2021_2025_mean,
    on=[
        "State_Standard",
        "District_Standard"
    ],
    how="left"
)

gw_features["GW_Recent_vs_Early_Change"] = (
    gw_features["GW_Mean_2021_2025"]
    - gw_features["GW_Mean_2015_2019"]
)


# =========================================================
# CLEAN FEATURE TABLE
# =========================================================

gw_features = (
    gw_features
    .replace([np.inf, -np.inf], np.nan)
    .reset_index(drop=True)
)


# =========================================================
# REPORT
# =========================================================

print("=" * 70)
print("GROUNDWATER TEMPORAL FEATURE TABLE")
print("=" * 70)

print(
    "\nShape:",
    gw_features.shape
)

print(
    "\nNumber of districts:",
    len(gw_features)
)

print(
    "\nColumns:"
)

print(
    gw_features.columns.tolist()
)

print(
    "\nMissing values:"
)

print(
    gw_features.isna().sum()
)

print(
    "\nFirst 10 districts:"
)

print(
    gw_features
    .head(10)
    .to_string(index=False)
)

GROUNDWATER TEMPORAL FEATURE TABLE

Shape: (717, 36)

Number of districts: 717

Columns:
['State_Standard', 'District_Standard', 'GW_1994_2025_Mean', 'GW_1994_2025_Median', 'GW_1994_2025_Std', 'GW_1994_2025_Min', 'GW_1994_2025_Max', 'GW_1994_2025_Years_Available', 'GW_1994_2025_Avg_Stations', 'GW_1994_2025_Trend', 'GW_2015_2025_Mean', 'GW_2015_2025_Median', 'GW_2015_2025_Std', 'GW_2015_2025_Min', 'GW_2015_2025_Max', 'GW_2015_2025_Years_Available', 'GW_2015_2025_Avg_Stations', 'GW_2015_2025_Trend', 'GW_2021_2025_Mean', 'GW_2021_2025_Median', 'GW_2021_2025_Std', 'GW_2021_2025_Min', 'GW_2021_2025_Max', 'GW_2021_2025_Years_Available', 'GW_2021_2025_Avg_Stations', 'GW_2021_2025_Trend', 'GW_Latest_2025', 'GW_Latest_Median_2025', 'GW_Latest_Min_2025', 'GW_Latest_Max_2025', 'GW_Latest_Std_2025', 'GW_Stations_2025', 'GW_Observations_2025', 'GW_Mean_2015_2019', 'GW_Mean_2021_2025', 'GW_Recent_vs_Early_Change']

Missing values:
State_Standard                   0
District_Standard                0

In [49]:
# =========================================================
# STEP 12 — VALIDATE GROUNDWATER TEMPORAL FEATURES
# =========================================================

trend_cols = [
    "GW_1994_2025_Trend",
    "GW_2015_2025_Trend",
    "GW_2021_2025_Trend"
]

change_cols = [
    "GW_Recent_vs_Early_Change"
]

print("=" * 70)
print("GROUNDWATER TEMPORAL FEATURE VALIDATION")
print("=" * 70)


# ---------------------------------------------------------
# 1. TREND DISTRIBUTIONS
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TREND DISTRIBUTIONS")
print("=" * 70)

for col in trend_cols:

    print(f"\n{col}")

    print(
        gw_features[col]
        .describe()
    )


# ---------------------------------------------------------
# 2. MOST POSITIVE / NEGATIVE LONG-TERM TRENDS
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("EXTREME LONG-TERM TRENDS")
print("=" * 70)

cols_to_show = [
    "State_Standard",
    "District_Standard",
    "GW_1994_2025_Years_Available",
    "GW_1994_2025_Mean",
    "GW_1994_2025_Trend"
]

print("\nMost positive trends:")

print(
    gw_features
    .sort_values(
        "GW_1994_2025_Trend",
        ascending=False
    )
    [cols_to_show]
    .head(15)
    .to_string(index=False)
)

print("\nMost negative trends:")

print(
    gw_features
    .sort_values(
        "GW_1994_2025_Trend",
        ascending=True
    )
    [cols_to_show]
    .head(15)
    .to_string(index=False)
)


# ---------------------------------------------------------
# 3. RECENT VS EARLY CHANGE
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("RECENT VS EARLY CHANGE")
print("=" * 70)

print(
    gw_features[
        "GW_Recent_vs_Early_Change"
    ].describe()
)

cols_to_show = [
    "State_Standard",
    "District_Standard",
    "GW_Mean_2015_2019",
    "GW_Mean_2021_2025",
    "GW_Recent_vs_Early_Change"
]

print("\nLargest positive changes:")

print(
    gw_features
    .sort_values(
        "GW_Recent_vs_Early_Change",
        ascending=False
    )
    [cols_to_show]
    .head(15)
    .to_string(index=False)
)

print("\nLargest negative changes:")

print(
    gw_features
    .sort_values(
        "GW_Recent_vs_Early_Change",
        ascending=True
    )
    [cols_to_show]
    .head(15)
    .to_string(index=False)
)


# ---------------------------------------------------------
# 4. HISTORICAL COVERAGE
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HISTORICAL COVERAGE")
print("=" * 70)

for col in [
    "GW_1994_2025_Years_Available",
    "GW_2015_2025_Years_Available",
    "GW_2021_2025_Years_Available"
]:

    print(f"\n{col}")

    print(
        gw_features[col]
        .describe()
    )


# ---------------------------------------------------------
# 5. DISTRICTS WITH VERY LOW COVERAGE
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("LOW COVERAGE DISTRICTS")
print("=" * 70)

low_coverage = gw_features[
    (
        gw_features[
            "GW_2015_2025_Years_Available"
        ] < 3
    )
]

print(
    f"\nDistricts with fewer than 3 "
    f"years during 2015–2025: "
    f"{len(low_coverage)}"
)

if len(low_coverage) > 0:

    print(
        low_coverage[
            [
                "State_Standard",
                "District_Standard",
                "GW_2015_2025_Years_Available",
                "GW_2021_2025_Years_Available",
                "GW_1994_2025_Years_Available"
            ]
        ]
        .sort_values(
            "GW_2015_2025_Years_Available"
        )
        .head(30)
        .to_string(index=False)
    )


# ---------------------------------------------------------
# 6. 2025 COVERAGE
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("2025 COVERAGE")
print("=" * 70)

missing_2025 = gw_features[
    gw_features["GW_Latest_2025"].isna()
]

print(
    f"\nDistricts without 2025 data: "
    f"{len(missing_2025)}"
)

if len(missing_2025) > 0:

    print(
        missing_2025[
            [
                "State_Standard",
                "District_Standard",
                "GW_2015_2025_Years_Available",
                "GW_2021_2025_Years_Available"
            ]
        ]
        .to_string(index=False)
    )


# ---------------------------------------------------------
# 7. INFINITE VALUES
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("INVALID VALUES")
print("=" * 70)

numeric_cols = (
    gw_features
    .select_dtypes(
        include=np.number
    )
    .columns
)

inf_count = np.isinf(
    gw_features[numeric_cols]
).sum().sum()

print(
    "Total infinite values:",
    inf_count
)

print(
    "Total NaN values:",
    gw_features.isna().sum().sum()
)

GROUNDWATER TEMPORAL FEATURE VALIDATION

TREND DISTRIBUTIONS

GW_1994_2025_Trend
count    713.000000
mean       0.098809
std        1.048550
min       -6.886364
25%       -0.047812
50%       -0.005904
75%        0.037207
max       21.285000
Name: GW_1994_2025_Trend, dtype: float64

GW_2015_2025_Trend
count    708.000000
mean       0.025239
std        1.120643
min       -6.886364
25%       -0.211406
50%       -0.075246
75%        0.035627
max       21.285000
Name: GW_2015_2025_Trend, dtype: float64

GW_2021_2025_Trend
count    707.000000
mean       0.157725
std        1.430479
min      -12.821333
25%       -0.188540
50%        0.070367
75%        0.414160
max       21.285000
Name: GW_2021_2025_Trend, dtype: float64

EXTREME LONG-TERM TRENDS

Most positive trends:
   State_Standard District_Standard  GW_1994_2025_Years_Available  GW_1994_2025_Mean  GW_1994_2025_Trend
      Uttarakhand      Rudra Prayag                             2          34.742500           21.285000
      Uttarakhand

In [50]:
# =========================================================
# STEP 13 — INSPECT EXTREME GROUNDWATER TRENDS
# =========================================================

# ---------------------------------------------------------
# Districts with suspiciously large long-term trends
# ---------------------------------------------------------

top_long_term = (
    gw_features
    .dropna(subset=["GW_1994_2025_Trend"])
    .sort_values(
        "GW_1994_2025_Trend",
        ascending=False
    )
    .head(10)
    [
        [
            "State_Standard",
            "District_Standard",
            "GW_1994_2025_Years_Available",
            "GW_1994_2025_Trend"
        ]
    ]
)

bottom_long_term = (
    gw_features
    .dropna(subset=["GW_1994_2025_Trend"])
    .sort_values(
        "GW_1994_2025_Trend",
        ascending=True
    )
    .head(10)
    [
        [
            "State_Standard",
            "District_Standard",
            "GW_1994_2025_Years_Available",
            "GW_1994_2025_Trend"
        ]
    ]
)

# ---------------------------------------------------------
# Combine districts for inspection
# ---------------------------------------------------------

inspect_districts = pd.concat(
    [
        top_long_term,
        bottom_long_term
    ]
).drop_duplicates(
    subset=[
        "State_Standard",
        "District_Standard"
    ]
)

# ---------------------------------------------------------
# Retrieve yearly groundwater values
# ---------------------------------------------------------

inspection = (
    district_year_gw
    .merge(
        inspect_districts[
            [
                "State_Standard",
                "District_Standard"
            ]
        ],
        on=[
            "State_Standard",
            "District_Standard"
        ],
        how="inner"
    )
    [
        [
            "State_Standard",
            "District_Standard",
            "Year",
            "GW_Mean",
            "GW_Median",
            "GW_Min",
            "GW_Max",
            "Station_Count"
        ]
    ]
    .sort_values(
        [
            "State_Standard",
            "District_Standard",
            "Year"
        ]
    )
)

print("=" * 70)
print("EXTREME LONG-TERM TREND INSPECTION")
print("=" * 70)

print(
    "\nNumber of districts inspected:",
    inspect_districts.shape[0]
)

print(
    "\nDistricts selected:"
)

print(
    inspect_districts
    .sort_values(
        "GW_1994_2025_Trend",
        ascending=False
    )
    .to_string(index=False)
)

print(
    "\n" + "=" * 70
)
print(
    "YEARLY VALUES"
)
print("=" * 70)

print(
    inspection
    .to_string(index=False)
)

EXTREME LONG-TERM TREND INSPECTION

Number of districts inspected: 20

Districts selected:
   State_Standard   District_Standard  GW_1994_2025_Years_Available  GW_1994_2025_Trend
      Uttarakhand        Rudra Prayag                             2           21.285000
      Uttarakhand             Chamoli                             2           13.285385
 Himachal Pradesh              Chamba                             2            4.100000
      Uttarakhand         Pithoragarh                             2            3.970182
      Uttarakhand       Tehri Garhwal                             3            3.677000
 Himachal Pradesh            Bilaspur                             2            2.882778
         Nagaland            Tseminyu                             2            2.546000
          Mizoram             Kolasib                             2            2.190000
Jammu And Kashmir              Poonch                             3            2.120000
          Mizoram            

In [51]:
# =========================================================
# STEP 14 — FINAL GROUNDWATER FEATURE TABLE
# =========================================================

gw_final = gw_features[
    [
        "State_Standard",
        "District_Standard",

        # Long-term groundwater condition
        "GW_1994_2025_Mean",
        "GW_1994_2025_Median",
        "GW_1994_2025_Std",
        "GW_1994_2025_Trend",
        "GW_1994_2025_Years_Available",

        # Modeling-period condition
        "GW_2015_2025_Mean",
        "GW_2015_2025_Median",
        "GW_2015_2025_Std",
        "GW_2015_2025_Trend",
        "GW_2015_2025_Years_Available",

        # Recent condition
        "GW_2021_2025_Mean",
        "GW_2021_2025_Median",
        "GW_2021_2025_Std",
        "GW_2021_2025_Trend",
        "GW_2021_2025_Years_Available",

        # Latest condition
        "GW_Latest_2025",
        "GW_Latest_Median_2025",
        "GW_Latest_Std_2025",
        "GW_Stations_2025",

        # Historical change
        "GW_Mean_2015_2019",
        "GW_Mean_2021_2025",
        "GW_Recent_vs_Early_Change"
    ]
].copy()


# ---------------------------------------------------------
# Add explicit coverage indicators
# ---------------------------------------------------------

gw_final["GW_2025_Available"] = (
    gw_final["GW_Latest_2025"]
    .notna()
    .astype(int)
)

gw_final["GW_2015_2025_Coverage_Ratio"] = (
    gw_final["GW_2015_2025_Years_Available"]
    / 11
)

gw_final["GW_2021_2025_Coverage_Ratio"] = (
    gw_final["GW_2021_2025_Years_Available"]
    / 5
)


# ---------------------------------------------------------
# Check duplicates
# ---------------------------------------------------------

duplicates = gw_final.duplicated(
    subset=[
        "State_Standard",
        "District_Standard"
    ]
).sum()


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("=" * 70)
print("FINAL GROUNDWATER FEATURE TABLE")
print("=" * 70)

print(
    "\nShape:",
    gw_final.shape
)

print(
    "\nDuplicate district keys:",
    duplicates
)

print(
    "\nColumns:"
)

print(
    gw_final.columns.tolist()
)

print(
    "\nMissing values:"
)

print(
    gw_final.isna().sum()
)

print(
    "\n2025 availability:"
)

print(
    gw_final[
        "GW_2025_Available"
    ].value_counts()
)

print(
    "\nCoverage ratio — 2015–2025:"
)

print(
    gw_final[
        "GW_2015_2025_Coverage_Ratio"
    ].describe()
)

print(
    "\nFirst 10 rows:"
)

print(
    gw_final
    .head(10)
    .to_string(index=False)
)

FINAL GROUNDWATER FEATURE TABLE

Shape: (717, 27)

Duplicate district keys: 0

Columns:
['State_Standard', 'District_Standard', 'GW_1994_2025_Mean', 'GW_1994_2025_Median', 'GW_1994_2025_Std', 'GW_1994_2025_Trend', 'GW_1994_2025_Years_Available', 'GW_2015_2025_Mean', 'GW_2015_2025_Median', 'GW_2015_2025_Std', 'GW_2015_2025_Trend', 'GW_2015_2025_Years_Available', 'GW_2021_2025_Mean', 'GW_2021_2025_Median', 'GW_2021_2025_Std', 'GW_2021_2025_Trend', 'GW_2021_2025_Years_Available', 'GW_Latest_2025', 'GW_Latest_Median_2025', 'GW_Latest_Std_2025', 'GW_Stations_2025', 'GW_Mean_2015_2019', 'GW_Mean_2021_2025', 'GW_Recent_vs_Early_Change', 'GW_2025_Available', 'GW_2015_2025_Coverage_Ratio', 'GW_2021_2025_Coverage_Ratio']

Missing values:
State_Standard                   0
District_Standard                0
GW_1994_2025_Mean                0
GW_1994_2025_Median              0
GW_1994_2025_Std                 4
GW_1994_2025_Trend               4
GW_1994_2025_Years_Available     0
GW_2015_2025_Mean

In [52]:
# Save final feature table to CSV
gw_final.to_csv("../clean/groundwater_features_final.csv", index=False)